In [1]:
import numpy as np
import random
from fractions import Fraction
import matplotlib.pyplot as plt
import math

In [2]:
n = 4            
N = 15            
a = 2          

R_sim = 2**4     

y_max = 5

x_register = np.arange(0, R_sim)

y_register = np.arange(-y_max, y_max + 1)

qubit_state = 0
z_mode_state = 0

initial_state = []

for x in x_register:
    for y in y_register:
        state = (int(x), int(y), z_mode_state, qubit_state)
        initial_state.append(state)

print(f"Number of basis states: {len(initial_state)}")
print(f"Sample states (x, y, z, qubit): {initial_state[:5]}")

Number of basis states: 176
Sample states (x, y, z, qubit): [(0, -5, 0, 0), (0, -4, 0, 0), (0, -3, 0, 0), (0, -2, 0, 0), (0, -1, 0, 0)]


In [3]:
def apply_M_N(state_list, N):
    new_state_list = []
    for (x, y, z, qubit) in state_list:
        new_y = N * y
        new_state_list.append((x, new_y, z, qubit))
    return new_state_list

after_M_N_state = apply_M_N(initial_state, N)

print(f"Sample states after M_N (x, y, z, qubit): {after_M_N_state[:5]}")

Sample states after M_N (x, y, z, qubit): [(0, -75, 0, 0), (0, -60, 0, 0), (0, -45, 0, 0), (0, -30, 0, 0), (0, -15, 0, 0)]


In [4]:
def apply_U_a_N_m(state_list, a, N):
    new_state_list = []
    for (x, y, z, qubit) in state_list:
        f_x = pow(a, x, N)  
        new_y = y + f_x
        new_state_list.append((x, new_y, z, qubit))
    return new_state_list

after_U_state = apply_U_a_N_m(after_M_N_state, a, N)

print(f"Sample states after U_a_N_m (x, y, z, qubit): {after_U_state[:5]}")


Sample states after U_a_N_m (x, y, z, qubit): [(0, -74, 0, 0), (0, -59, 0, 0), (0, -44, 0, 0), (0, -29, 0, 0), (0, -14, 0, 0)]


In [5]:
def measure_mode_B(state_list, N):
    y_mod_N = [(x, y % N, z, qubit) for (x, y, z, qubit) in state_list]
    
    possible_k = [y for (x, y, z, qubit) in y_mod_N]

    measurement_k = random.choice(possible_k)

    post_measurement_states = []
    for (x, y, z, qubit) in state_list:
        if (y % N) == measurement_k:
            post_measurement_states.append((x, y, z, qubit))
    
    return measurement_k, post_measurement_states

measured_k, post_meas_state = measure_mode_B(after_U_state, N)

print(f"Measurement outcome k (mod N): {measured_k}")
print(f"Number of post-measurement states: {len(post_meas_state)}")
print(f"Sample surviving states (x, y, z, qubit): {post_meas_state[:5]}")

Measurement outcome k (mod N): 8
Number of post-measurement states: 44
Sample surviving states (x, y, z, qubit): [(3, -67, 0, 0), (3, -52, 0, 0), (3, -37, 0, 0), (3, -22, 0, 0), (3, -7, 0, 0)]


In [11]:
def fourier_transform_and_measure(state_list, R):
    x_values = [x for (x, y, z, qubit) in state_list]
    amplitudes = np.zeros(R, dtype=complex)
    
    for x in x_values:
        amplitudes[x] = 1.0 

    norm = np.linalg.norm(amplitudes)
    if norm > 0:
        amplitudes /= norm

    ft_amplitudes = np.fft.fft(amplitudes)

    probabilities = np.abs(ft_amplitudes)**2
    probabilities /= np.sum(probabilities)  
    
    measured_p = np.random.choice(np.arange(R), p=probabilities)
    
    return measured_p, probabilities

measured_p, p_distribution = fourier_transform_and_measure(post_meas_state, R_sim)


In [12]:
def continued_fraction_expansion(x, max_denominator):
    return Fraction(x).limit_denominator(max_denominator)

def find_candidate_period(measured_p, R, N):
    if measured_p == 0:
        return None  

    frac = continued_fraction_expansion(measured_p / R, N)
    candidate_r = frac.denominator
    
    return candidate_r

candidate_r = find_candidate_period(measured_p, R_sim, N)

print(f"Estimated candidate period r: {candidate_r}")

Estimated candidate period r: 4


In [13]:
def try_find_factors(N, r, a):
    if r % 2 != 0:
        return None
    candidate = pow(a, r//2, N)
    if candidate == N-1 or candidate == 1:
        return None
    p = np.gcd(candidate-1, N)
    q = np.gcd(candidate+1, N)
    if p*q == N and p != 1 and q != 1:
        return (p, q)
    else:
        return None
    
factors = try_find_factors(N, candidate_r, a)
print(f"Found factors: {factors}")

Found factors: (np.int64(3), np.int64(5))


In [14]:
def pick_random_coprime(N):
    while True:
        a = random.randint(2, N-1)
        if math.gcd(a, N) == 1:
            return a

In [15]:
def run_multiple_trials(N, num_trials=100):
    success_count = 0
    all_factors = []

    n = math.ceil(math.log2(N))
    R=2**12
    y_max = int(np.ceil(np.log(N))) + 2  

    print(f"Using R = {R}, y_max = {y_max}")

    for trial in range(num_trials):
        a = pick_random_coprime(N)

        x_register = np.arange(0, R)
        y_register = np.arange(-y_max, y_max + 1)

        qubit_state = 0
        z_mode_state = 0

        initial_state = []
        for x in x_register:
            for y in y_register:
                state = (int(x), int(y), z_mode_state, qubit_state)
                initial_state.append(state)

        state = initial_state
        after_MN_state = apply_M_N(state, N)
        after_U_state = apply_U_a_N_m(after_MN_state, a, N)
        measured_k, post_meas_state = measure_mode_B(after_U_state, N)
        
        if not post_meas_state:
            continue
        
        measured_p_idx, _ = fourier_transform_and_measure(post_meas_state, R)
        candidate_r = find_candidate_period(measured_p_idx, R, N)
        
        if candidate_r is None:
            continue
        
        factors = try_find_factors(N, candidate_r, a)
        
        if factors:
            success_count += 1
            all_factors.append(factors)
    
    success_rate = success_count / num_trials
    return success_count, success_rate, all_factors


N = 85 
success_count, success_rate, factors_found = run_multiple_trials(N=N, num_trials=500)

print(f"Number of successes: {success_count}")
print(f"Success rate: {success_rate:.2f}")
print(f"Factors found: {factors_found}")

Using R = 4096, y_max = 7
Number of successes: 208
Success rate: 0.42
Factors found: [(np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(17), np.int64(5)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(17), np.int64(5)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), np.int64(17)), (np.int64(5), n